# Statim: grounded synthetic data, Colab test

Runs the same pilot as on the local RTX 3070, on a free Colab GPU:
**Qwen/Qwen3-8B** (Apache-2.0) writes, **microsoft/Phi-4-mini-instruct** (MIT) checks blind,
seeds come from licence-clean corpora (BillSum CC0, GovReport CC-BY-4.0, EUR-Lex-Sum CC-BY-4.0).

1. *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
2. Colab has no held-out evaluation texts, so the leakage check is **deferred**: the output is
   marked `checked: false` and must be filtered on pop-os with `tools/synth/leakage.py --filter`
   before any item is used for training.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Ollama (official install script; it needs zstd), server with 8 parallel slots
!apt-get -qq update && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1; tail -n 2 /tmp/ollama-install.log
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_NUM_PARALLEL="8", OLLAMA_FLASH_ATTENTION="1",
           OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="60m")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode()); break
    except Exception:
        time.sleep(1)

In [ ]:
!ollama pull qwen3:8b 2>&1 | tail -n 1
!ollama pull phi4-mini 2>&1 | tail -n 1
!ollama list
# the T4 must be used: a short generation, then the server log must show CUDA offload
!ollama run qwen3:8b "Say ok" --verbose 2>&1 | tail -n 3
!grep -iE "offloaded|CUDA|compute capability" /tmp/ollama.log | tail -n 3

In [ ]:
!rm -rf /content/statim && git clone -q --depth 1 -b data/synth-v2 https://github.com/BEKO2210/statim /content/statim
!pip -q install datasets huggingface_hub pyarrow
%cd /content/statim
!git log --oneline -1

## 1. Throughput (24 kept items per capability at 1, 2, 4, 8 parallel requests)

In [ ]:
!TMPDIR=/tmp python tools/synth/bench_throughput.py --items 24 --cache /tmp/statim-synth-corpora --defer-leakage-check

## 2. Pilot: 200 kept items per capability (urgency + NLI in 8 languages)

In [ ]:
CONCURRENCY = 8  # set to the best value from step 1
!TMPDIR=/tmp python tools/synth/grounded.py --per-capability 200 --concurrency {CONCURRENCY} \
    --cache /tmp/statim-synth-corpora --out-dir /content/synth-colab-pilot --defer-leakage-check
!head -c 3000 /content/synth-colab-pilot/samples.md

## 3. Save the result (unchecked until filtered on pop-os)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/statim-synth && cp -r /content/synth-colab-pilot /content/drive/MyDrive/statim-synth/
!ls -la /content/drive/MyDrive/statim-synth/synth-colab-pilot